# Repeated sampling experiments

Independent repeated samples at a fixed sample size: the thesis histograms, the calibrated spectral selector, and the custom-beta Monte Carlo study.

Run the cells in order; all required functions are imported below. Every displayed figure is saved automatically under `outputs/pdfs/04_repeated_sampling_experiments/`.

Source: [KDE chapter](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).


In [ ]:
from _setup import setup_notebook
from kde_tools.spectral_experiments import plot_discrete_bandwidth_distribution
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from IPython.display import display
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.distribution_runs import run_beta2_distribution
from kde_tools.energy_minimization import EnergySearchConfig, minimize_bandwidth_energy
from kde_tools.fourier.similarities import (
    DEFAULT_FOURIER_GRID_TAG,
    DEFAULT_FREQUENCY_CUTOFF,
    DEFAULT_FREQUENCY_INTERVALS,
    FOURIER_L2_METHOD_VERSION,
    beta2_divergence_matrix_fourier,
)
from kde_tools.joint_experiments import draw_sample
from kde_tools.perron import PERRON_SOLVER_VERSION, perron_peak_index, perron_vector_from_divergence
from kde_tools.qhat import expected_gaussian_qhat_log_energy
from kde_tools.spectral_sweeps import run_l2_point
from matplotlib.ticker import FormatStrFormatter, MultipleLocator

paths = setup_notebook("04_repeated_sampling_experiments")
code_directory = paths.code
pdf_output_directory = paths.pdfs
seed = 20260818
kde_thesis_figure_dir = code_directory / "outputs/images/thesis"
kde_thesis_figure_dir.mkdir(parents=True, exist_ok=True)
ENERGY_SEARCH = EnergySearchConfig()
N_DATA = 200
H_MIN, H_MAX = (0.01, 2.0)
BETA = 2.0
FREQUENCY_CUTOFF = DEFAULT_FREQUENCY_CUTOFF
FREQUENCY_INTERVALS = DEFAULT_FREQUENCY_INTERVALS
FFT_HALF_WIDTH = np.pi * FREQUENCY_INTERVALS / FREQUENCY_CUTOFF
FFT_SPATIAL_POINTS = 2 * FREQUENCY_INTERVALS
plt.style.use("default")


## 2. Configurable beta-2 selection histogram

Choose `beta2_profile = "figure_3_5"` for the original lambda = 1 comparison, or
`beta2_profile = "localized"` for lambda = N^(3/2). Both profiles use the same
Gaussian samples and three grid selectors: spectral, ISE, and Q-hat.

The default n = 200, N = 1000, M = 1000 and seed reproduce the earlier
experiments. Both profiles now save resumable checkpoints, selections, histogram
frequencies, and per-repetition eigenpair residual/gap diagnostics. Runs are
separated by profile and complete configuration under
`outputs/data/spectral_scale_selection_kde/`; compatible earlier checkpoints
can be imported after checking settings, grid, sample hashes, and method versions.
Historical results remain in their original directories.

Run the next three cells after changing the profile. The localized profile
replaces the former standalone localized-selection notebook. The Figure 3.5
profile updates its thesis figure; the localized profile saves its own PNG and
automatic PDF. Sample-size sweeps are in [05_increasing_sample_size_experiments.ipynb](05_increasing_sample_size_experiments.ipynb).

Sources: [shared experiment runner](../kde_tools/distribution_runs.py);
[Perron solver and diagnostics](../kde_tools/perron.py);
[saved original localized settings](../outputs/data/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000/settings.json).


The `exact_gaussian` profile replaces the former Gaussian L²/ISE notebook. It uses its original independent seeds (`910000 + repetition`), exact pair divergence, and preview/thesis grids. It exports selected ISE values, an overlay, and the two individual distributions. Choose the profile in the next cell and rerun these three cells.


In [ ]:
beta2_profile = "figure_3_5"
beta2_n = 200
beta2_N = 1000
beta2_M = 1000
beta2_binning = 5
beta2_h_min = 0.01
beta2_h_max = 2.0
if beta2_profile not in {"figure_3_5", "localized"}:
    raise ValueError("Choose figure_3_5 or localized for beta2_profile.")
beta2_lambda = 1.0 if beta2_profile != "localized" else beta2_N ** (3.0 / 2.0)
beta2_seed = seed + 10
beta2_sigma2 = 1.0
beta2_show_mise_reference = True
beta2_frequency_cutoff = DEFAULT_FREQUENCY_CUTOFF
beta2_frequency_intervals = DEFAULT_FREQUENCY_INTERVALS
beta2_half_width = np.pi * beta2_frequency_intervals / beta2_frequency_cutoff
beta2_spatial_points = 2 * beta2_frequency_intervals
beta2_validate_fft = True
beta2_workers = 4
beta2_reuse_checkpoints = True
beta2_data_root = code_directory / "outputs/data/spectral_scale_selection_kde"
beta2_image_root = code_directory / "outputs/images/spectral_scale_selection_kde"
beta2_image_directory = beta2_image_root / beta2_profile
if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_binning < 1 or int(beta2_binning) != beta2_binning:
    raise ValueError("beta2_binning must be a positive integer.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_lambda <= 0.0:
    raise ValueError("beta2_lambda must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2:
    raise ValueError("beta2_spatial_points must be at least 2.")
beta2_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_h_min, h_max=beta2_h_max, beta=2.0, grid_size=beta2_N
)
print(f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, lambda={beta2_lambda:g}")


beta=2, n=200, N=1000, M=1000, lambda=1


In [ ]:
beta2_legacy_names = (
    ["spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000"]
    if beta2_profile == "localized"
    else ["figure_3_5_bandwidth_distribution_n1000"]
)
beta2_import_directories = [
    directory
    for name in beta2_legacy_names
    for directory in (
        code_directory / "outputs/data" / name / FOURIER_L2_METHOD_VERSION,
        code_directory / "outputs/data" / name,
    )
]
beta2_control_directories = [
    code_directory
    / "outputs/data/figure_3_5_bandwidth_distribution_n1000"
    / FOURIER_L2_METHOD_VERSION,
    code_directory / "outputs/data/figure_3_5_bandwidth_distribution_n1000",
]
beta2_run = run_beta2_distribution(
    beta2_h_grid,
    n=beta2_n,
    repetitions=beta2_M,
    lambda_value=beta2_lambda,
    seed=beta2_seed,
    data_root=beta2_data_root,
    profile=beta2_profile,
    half_width=beta2_half_width,
    spatial_points=beta2_spatial_points,
    workers=beta2_workers,
    reuse=beta2_reuse_checkpoints,
    import_directories=beta2_import_directories,
    control_directories=beta2_control_directories,
    validate_fourier=beta2_validate_fft,
    binning=beta2_binning,
    sigma2=beta2_sigma2,
)
beta2_selected_indices = beta2_run["selected_indices"][:, 0]
beta2_ise_selected_indices = beta2_run["selected_indices"][:, 1]
beta2_selection_counts, beta2_ise_selection_counts = beta2_run["counts"][:2]
beta2_qhat_selected_indices = beta2_run["selected_indices"][:, 2]
beta2_qhat_selection_counts = beta2_run["counts"][2]
beta2_solver_diagnostics = beta2_run["solver_diagnostics"]
beta2_run_directory = beta2_run["run_directory"]
beta2_image_directory = beta2_image_root / beta2_profile / beta2_run["configuration_id"]
print(f"Saved selections and eigenpair diagnostics: {beta2_run_directory}")
print(f"Maximum relative residual: {beta2_solver_diagnostics[:, 0].max():.3e}")
print(f"Minimum relative spectral gap: {beta2_solver_diagnostics[:, 1].min():.3e}")


figure_3_5: 0/1000 repetitions cached; /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/data/spectral_scale_selection_kde/figure_3_5/242dad92c362bd38
figure_3_5: completed 100/1000


In [ ]:
beta2_histogram_counts = np.stack(
    [beta2_selection_counts, beta2_ise_selection_counts, beta2_qhat_selection_counts]
)
assert beta2_histogram_counts.shape == (3, beta2_N)
beta2_bin_starts = np.arange(0, beta2_N, beta2_binning)
beta2_bin_stops = np.minimum(beta2_bin_starts + beta2_binning, beta2_N)
beta2_bin_centers = (beta2_bin_starts + beta2_bin_stops - 1) / 2.0
beta2_bin_widths = beta2_bin_stops - beta2_bin_starts
beta2_binned_counts = np.add.reduceat(beta2_histogram_counts, beta2_bin_starts, axis=1)
assert np.array_equal(beta2_binned_counts.sum(axis=1), beta2_histogram_counts.sum(axis=1))
assert np.all(beta2_binned_counts.sum(axis=1) == beta2_M)
beta2_distribution_names = [
    "spectral selections",
    "ISE-optimal grid bandwidths",
    "Q-hat minimizers",
]
beta2_histogram, beta2_axis = plt.subplots(figsize=(12.0, 6.32))
beta2_histogram.subplots_adjust(left=0.085, right=0.98, top=0.97, bottom=0.15)
for beta2_label, beta2_color, beta2_opacity, beta2_values in zip(
    beta2_distribution_names,
    ["#C65D00", "#555555", "#43877E"],
    [0.7, 0.7, 0.55],
    beta2_binned_counts,
):
    beta2_nonzero = np.flatnonzero(beta2_values)
    beta2_axis.bar(
        beta2_bin_centers[beta2_nonzero],
        beta2_values[beta2_nonzero] / beta2_M,
        width=0.94 * beta2_bin_widths[beta2_nonzero],
        color=beta2_color,
        alpha=beta2_opacity,
        edgecolor="white",
        linewidth=0.3,
        label=beta2_label,
    )
beta2_mise_bandwidth = gaussian_mise_oracle_bandwidth(beta2_n)
beta2_grid_indices = np.arange(beta2_N, dtype=float)
beta2_mise_grid_position = float(
    np.interp(np.log(beta2_mise_bandwidth), np.log(beta2_h_grid), beta2_grid_indices)
)
beta2_reference_label = f"Gaussian MISE optimum: h={beta2_mise_bandwidth:.4g}"
beta2_axis.axvline(
    beta2_mise_grid_position,
    color="tab:red",
    linestyle="--",
    linewidth=1.8,
    label=beta2_reference_label,
)
beta2_ticks = np.unique(np.linspace(0, beta2_N - 1, 7, dtype=int))
beta2_axis.set_xticks(beta2_ticks, [f"{beta2_h_grid[i]:.3g}" for i in beta2_ticks])
beta2_axis.set_xlim(-0.5, beta2_N - 0.5)
beta2_axis.set_ylim(0, 1.05 * float(beta2_binned_counts.max()) / beta2_M)
beta2_axis.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
beta2_axis.yaxis.set_major_locator(MultipleLocator(0.01))
beta2_axis.set_xlabel("chosen bandwidth $h_{k_N}$", fontsize=12, labelpad=12, color="#304967")
beta2_axis.set_ylabel("relative frequency", fontsize=12, labelpad=12, color="#304967")
beta2_axis.tick_params(axis="both", labelsize=10, colors="#304967", length=0)
beta2_axis.grid(axis="y", color="#E5ECF6", linewidth=0.6)
beta2_axis.set_axisbelow(True)
for beta2_side in ("top", "right", "left"):
    beta2_axis.spines[beta2_side].set_visible(False)
beta2_axis.spines["bottom"].set_color("#E5ECF6")
beta2_image_directory.mkdir(parents=True, exist_ok=True)
beta2_png_path = beta2_image_directory / f"{beta2_profile}_N{beta2_N}_bins{beta2_binning}.png"
beta2_histogram.savefig(beta2_png_path, dpi=180, facecolor="white")
print(f"Saved comparison figure: {beta2_png_path}")
if beta2_profile == "figure_3_5":
    beta2_thesis_figure_path = kde_thesis_figure_dir / "distribution_of_chosen_bandwidths.pdf"
    beta2_histogram.savefig(beta2_thesis_figure_path, format="pdf", bbox_inches="tight")
    print(f"Updated Figure 3.5: {beta2_thesis_figure_path}")
print(f"N={beta2_N}; binning={beta2_binning}; bins={len(beta2_bin_starts)}")
if beta2_profile == "localized" and (beta2_n, beta2_N, beta2_M) == (200, 1000, 1000):
    beta2_localized_delivery = (
        code_directory
        / "outputs/pdfs/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000/001_bandwidth_distributions_5_consecutive_grid_points_per_bin_n_1000_n_200_m_1000_la.pdf"
    )
    beta2_localized_delivery.parent.mkdir(parents=True, exist_ok=True)
    beta2_histogram.savefig(beta2_localized_delivery, format="pdf", bbox_inches="tight")
plt.show()
plt.close(beta2_histogram)


NameError: name 'np' is not defined

## Follow-up experiment: sampling distribution at calibrated $(N,\lambda)$

The fixed-sample experiment shows that the admissible choice $(N,\lambda_N)=(1000,1000)$ is already close to the continuous sample-conditional selector. We now freeze these numerical parameters and draw $M=1000$ independent samples of size $n=200$ from $N(0,1)$. Thus, only the data vary across repetitions.

For each sample we compute the same beta-2 spectral selector on the same geometric bandwidth grid. The resulting histogram estimates the finite-$n$ sampling distribution of $\widehat h_{1000,1000}$. For comparison, the numerical summary reports $h_{\mathbb E Q}=\arg\min_h\mathbb E[\widehat Q_{n,2}(h)]$, rather than the expectation of the random minimizer.

Every repetition uses the centralized stabilized `eigh` solver in `kde_tools.perron`. The first repetition reports its eigenpair residual and spectral gap; the compatibility matrix-vector count is zero for this direct solver.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
REPEATED_SAMPLE_M = 100
REPEATED_SAMPLE_N = 1000
REPEATED_SAMPLE_LAMBDA = 1000.0
REPEATED_SAMPLE_SEED = 20260903
repeated_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=BETA, grid_size=REPEATED_SAMPLE_N
)
joint_spectral_cache_root = code_directory / "outputs/data/shared_spectral_sweeps"
repeated_samples_bank = np.random.default_rng(REPEATED_SAMPLE_SEED).normal(
    0.0, 1.0, size=(REPEATED_SAMPLE_M, N_DATA)
)
repeated_settings = {
    "repetitions": REPEATED_SAMPLE_M,
    "maximum_sample_size": N_DATA,
    "seed": REPEATED_SAMPLE_SEED,
    "sigma2": 1.0,
    "fft_half_width": FFT_HALF_WIDTH,
    "fft_spatial_points": FFT_SPATIAL_POINTS,
}
repeated_start_time = time.perf_counter()
repeated_spectral_result = run_l2_point(
    N_DATA,
    repeated_samples_bank,
    repeated_h_grid,
    repeated_settings,
    [REPEATED_SAMPLE_LAMBDA],
    pdf_output_directory.parents[1]
    / "data/experiment_1_joint_N_lambda_limit/shared_runner"
    / DEFAULT_FOURIER_GRID_TAG
    / f"repeated_N{REPEATED_SAMPLE_N}_M{REPEATED_SAMPLE_M}_seed{REPEATED_SAMPLE_SEED}",
    cache_root=joint_spectral_cache_root,
    diagnostics=True,
)
repeated_selected_index = repeated_spectral_result["selected_indices"][:, 0]
repeated_selected_h = repeated_h_grid[repeated_selected_index]
repeated_diagnostics = repeated_spectral_result["solver_diagnostics"][:, 0]
repeated_solver_matvecs = repeated_diagnostics[:, 2].astype(int)
solver_validation = {
    "selected index": int(repeated_selected_index[0]),
    "selected bandwidth": float(repeated_selected_h[0]),
    "relative residual": float(repeated_diagnostics[0, 0]),
    "relative spectral gap": float(repeated_diagnostics[0, 1]),
    "solver matvecs": int(repeated_solver_matvecs[0]),
}
repeated_elapsed = time.perf_counter() - repeated_start_time
repeated_sample_results = pd.DataFrame(
    {
        "repetition": np.arange(1, REPEATED_SAMPLE_M + 1),
        "selected_index": repeated_selected_index,
        "selected_h": repeated_selected_h,
        "solver_matvecs": repeated_solver_matvecs,
        "n": N_DATA,
        "M": REPEATED_SAMPLE_M,
        "N": REPEATED_SAMPLE_N,
        "lambda": REPEATED_SAMPLE_LAMBDA,
        "seed": REPEATED_SAMPLE_SEED,
        "beta": BETA,
        "h_min": H_MIN,
        "h_max": H_MAX,
        "fft_half_width": FFT_HALF_WIDTH,
        "fft_spatial_points": FFT_SPATIAL_POINTS,
        "l2_method": FOURIER_L2_METHOD_VERSION,
        "perron_solver": PERRON_SOLVER_VERSION,
    }
)
repeated_data_directory = (
    pdf_output_directory.parents[1]
    / "data"
    / "experiment_1_joint_N_lambda_limit"
    / PERRON_SOLVER_VERSION
    / FOURIER_L2_METHOD_VERSION
    / DEFAULT_FOURIER_GRID_TAG
)
repeated_data_directory.mkdir(parents=True, exist_ok=True)
repeated_results_path = (
    repeated_data_directory
    / f"repeated_sample_selections_n{N_DATA}_N{REPEATED_SAMPLE_N}_lambda{REPEATED_SAMPLE_LAMBDA:g}_M{REPEATED_SAMPLE_M}.csv"
)
repeated_grid_path = repeated_data_directory / f"bandwidth_grid_N{REPEATED_SAMPLE_N}.csv"
repeated_sample_results.to_csv(repeated_results_path, index=False)
pd.DataFrame({"grid_index": np.arange(REPEATED_SAMPLE_N), "bandwidth_h": repeated_h_grid}).to_csv(
    repeated_grid_path, index=False
)
print(f"Completed all {REPEATED_SAMPLE_M} repetitions in {repeated_elapsed:.1f} seconds.")
print(f"Saved repetition data: {repeated_results_path}")
print(f"Saved bandwidth grid: {repeated_grid_path}")
display(pd.DataFrame([solver_validation]).round(10))


In [ ]:
expected_q_minimum = minimize_bandwidth_energy(
    lambda z: expected_gaussian_qhat_log_energy(z, N_DATA), H_MIN, H_MAX, config=ENERGY_SEARCH
)
if expected_q_minimum["diagnostics"]["resolution_stable"] is False:
    raise ArithmeticError("Energy search did not stabilize; increase ENERGY_SEARCH resolution.")
h_expected_q = expected_q_minimum["global_minimum"]["h"]
sampling_summary = pd.DataFrame(
    {
        "n": [N_DATA],
        "M": [REPEATED_SAMPLE_M],
        "N": [REPEATED_SAMPLE_N],
        "lambda": [REPEATED_SAMPLE_LAMBDA],
        "mean selected h": [repeated_selected_h.mean()],
        "median selected h": [np.median(repeated_selected_h)],
        "sd selected h": [repeated_selected_h.std(ddof=1)],
        "5% quantile": [np.quantile(repeated_selected_h, 0.05)],
        "95% quantile": [np.quantile(repeated_selected_h, 0.95)],
        "argmin E[Q_hat]": [h_expected_q],
        "mean solver matvecs": [repeated_solver_matvecs.mean()],
        "elapsed seconds": [repeated_elapsed],
    }
)
display(sampling_summary.round(6))
sampling_summary_path = (
    repeated_data_directory
    / f"repeated_sample_summary_n{N_DATA}_N{REPEATED_SAMPLE_N}_lambda{REPEATED_SAMPLE_LAMBDA:g}_M{REPEATED_SAMPLE_M}.csv"
)
sampling_summary.to_csv(sampling_summary_path, index=False)
print(f"Saved summary data: {sampling_summary_path}")
repeated_selection_counts = np.bincount(repeated_selected_index, minlength=REPEATED_SAMPLE_N)
repeated_histogram, repeated_axis = plot_discrete_bandwidth_distribution(
    repeated_h_grid, repeated_selection_counts,
    title="Sampling distribution of the calibrated spectral selector",
)
plt.show()
plt.close(repeated_histogram)


### Interpretation of the repeated-sample experiment

Increasing $M$ estimates the sampling distribution more accurately; it does not alter the selector. Because $n=200$ remains fixed, sampling variability persists even though the numerical grid and localization parameters have already been calibrated. The comparison with $h_{\mathbb E Q}$ tests whether the distribution is approximately centered near the minimizer of the expected criterion. Equality is not automatic, since expectation and minimization need not commute.


## Custom-beta Monte Carlo distribution

Fixed sample size with independent draws, followed by a histogram, empirical CDF, and grid-frequency display.


In [ ]:
MC_BETA = 1 / 7
MC_N = 1000
MC_ALPHA = 1.9
MC_LAMBDA = float(MC_N**MC_ALPHA)
MC_SAMPLE_SIZE = 200
MC_REPETITIONS = 2000
MC_SEED = 12345
mc_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=H_MIN, h_max=H_MAX, beta=MC_BETA, grid_size=MC_N
)
rng = np.random.default_rng(MC_SEED)
mc_rows = []
mc_start_time = time.perf_counter()
for repetition in range(1, MC_REPETITIONS + 1):
    current_samples = draw_sample(rng, MC_SAMPLE_SIZE)
    current_divergence = beta2_divergence_matrix_fourier(
        current_samples, mc_h_grid, half_width=FFT_HALF_WIDTH, spatial_points=FFT_SPATIAL_POINTS
    )
    current_vector = perron_vector_from_divergence(current_divergence, MC_LAMBDA)
    selected_index = int(perron_peak_index(current_vector))
    selected_h = float(mc_h_grid[selected_index])
    mc_rows.append(
        {
            "repetition": repetition,
            "n": MC_SAMPLE_SIZE,
            "N": MC_N,
            "beta": MC_BETA,
            "alpha": MC_ALPHA,
            "lambda": MC_LAMBDA,
            "selected_index": selected_index,
            "selected_h": selected_h,
        }
    )
    if repetition % 10 == 0:
        elapsed = time.perf_counter() - mc_start_time
        print(
            f"Completed {repetition:3d}/{MC_REPETITIONS} repetitions after {elapsed:.1f} seconds."
        )
mc_bandwidth_results = pd.DataFrame(mc_rows)
mc_elapsed = time.perf_counter() - mc_start_time
print()
print(f"Completed all {MC_REPETITIONS} repetitions in {mc_elapsed:.2f} seconds.")
print(f"beta     = {MC_BETA:.8f}")
print(f"N        = {MC_N}")
print(f"alpha    = {MC_ALPHA}")
print(f"lambda_N = {MC_LAMBDA:.6e}")
selected_bandwidths = mc_bandwidth_results["selected_h"].to_numpy()
mc_mean = float(np.mean(selected_bandwidths))
mc_median = float(np.median(selected_bandwidths))
mc_std = float(np.std(selected_bandwidths, ddof=1))
mc_q025, mc_q25, mc_q75, mc_q975 = np.quantile(selected_bandwidths, [0.025, 0.25, 0.75, 0.975])
print()
print("Selected bandwidth distribution")
print("--------------------------------")
print(f"mean       = {mc_mean:.8f}")
print(f"median     = {mc_median:.8f}")
print(f"std        = {mc_std:.8f}")
print(f"2.5%       = {mc_q025:.8f}")
print(f"25%        = {mc_q25:.8f}")
print(f"75%        = {mc_q75:.8f}")
print(f"97.5%      = {mc_q975:.8f}")
display(mc_bandwidth_results)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(selected_bandwidths, bins=20, density=True, alpha=0.75, edgecolor="black")
ax.axvline(mc_mean, linestyle="--", linewidth=1.8, label=f"mean $={mc_mean:.4f}$")
ax.axvline(mc_median, linestyle=":", linewidth=1.8, label=f"median $={mc_median:.4f}$")
ax.set_xlabel("selected bandwidth $\\hat h_{\\rm spec}$")
ax.set_ylabel("density")
ax.set_title(
    f"Monte Carlo distribution of $\\hat h_{{\\rm spec}}$\n$n={MC_SAMPLE_SIZE}$, $N={MC_N}$, $\\beta=1/7$, $\\lambda_N=N^{{{MC_ALPHA}}}$"
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
sorted_h = np.sort(selected_bandwidths)
empirical_cdf = np.arange(1, MC_REPETITIONS + 1) / MC_REPETITIONS
fig, ax = plt.subplots(figsize=(10, 6))
ax.step(sorted_h, empirical_cdf, where="post", linewidth=1.8)
ax.set_xlabel("selected bandwidth $\\hat h_{\\rm spec}$")
ax.set_ylabel("empirical CDF")
ax.set_title(f"Empirical distribution of $\\hat h_{{\\rm spec}}$, $n={MC_SAMPLE_SIZE}$")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
selected_indices = mc_bandwidth_results["selected_index"].to_numpy(dtype=int)
selected_counts = np.bincount(selected_indices, minlength=len(mc_h_grid)).astype(float)
mean_index = int(np.argmin(np.abs(mc_h_grid - mc_mean)))
median_index = int(np.argmin(np.abs(mc_h_grid - mc_median)))
reference_lines = [
    (f"mean $\\hat h_{{\\rm spec}}={mc_mean:.4f}$", mean_index, "#D55E00"),
    (f"median $\\hat h_{{\\rm spec}}={mc_median:.4f}$", median_index, "#009E73"),
]
fig, ax = plot_discrete_bandwidth_distribution(
    mc_h_grid, selected_counts,
    label=rf"$\beta=1/7,\ N={MC_N},\ \lambda_N=N^{{{MC_ALPHA}}}$",
    reference_lines=reference_lines, tick_count=9, figsize=(11, 6),
    title=f"Distribution over {MC_REPETITIONS} samples; n={MC_SAMPLE_SIZE}, N={MC_N}",
)
plt.show()
plt.close(fig)
